# Step 3 — Is the pitchfork generic? Network-free bifurcation analysis on real data (MNIST)

**Key identity (validated in Step 2).** For a fixed direction `u`, the conditional drift `E[u·score(x_t) | m]` equals the score of the *marginal* of `m = u·x_t`, and that marginal is the data's projection convolved with `N(0, sigma^2)`. So the bifurcation diagram along a class direction is a property of the **data**, and we can compute it exactly with a 1-D convolution. No network needed.

**Hypothesis.** The symmetric pitchfork requires the two projected class distributions to be mirror images. Real classes differ in shape and width, so even with balanced weights the bifurcation should be an *imperfect* pitchfork, which unfolds into a **fold** (saddle-node): the minor mode is born at a finite |m| rather than at 0.

**Diagnostic.** Track the stationary points of `p_tau(m)` as tau decreases. Record the **birth jump** = distance between the newly born max/min pair and the stable fixed point that existed just before. Pitchfork: the existing maximum itself splits, jump = 0. Fold: the existing maximum survives and the new pair appears elsewhere, jump > 0 (in the Step 1 control: 0 for w=0.5, about 1.7 for w=0.8).

Setup: digit pairs (0,1), (3,8), (4,9); balanced and w = 0.8; resolutions 28, 14, 7 (average pooling). Order parameter scaled so class means sit at ±1; `tau = 4 sigma^2 / ||mu_a - mu_b||^2`.

**Kaggle:** enable Internet in the notebook settings so torchvision can download MNIST. CPU is enough (~3 min).

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
from torchvision import datasets
import torch.nn.functional as F
rng = np.random.default_rng(0)
plt.rcParams['figure.dpi'] = 110

ds = datasets.MNIST('/kaggle/working/mnist', train=True, download=True)
X = ds.data.numpy().astype(np.float32) / 127.5 - 1.0     # (60000, 28, 28) in [-1, 1]
Y = ds.targets.numpy()
print(X.shape, Y.shape)

def pool(x, res):
    t = torch.from_numpy(x)[:, None]
    if res != 28:
        t = F.adaptive_avg_pool2d(t, res)
    return t[:, 0].reshape(len(x), -1).numpy()

## Core: exact 1-D marginal of the order parameter and its stationary points

In [ ]:
GRID = np.linspace(-4, 4, 4001)
DG = GRID[1] - GRID[0]

def order_param(Xa, Xb, w):
    """Project a w/(1-w) mixture of classes a,b onto the class direction, scaled so means are at +/-1.
    Returns m0 samples and ||Delta||^2 (needed to convert sigma -> tau)."""
    mu_a, mu_b = Xa.mean(0), Xb.mean(0)
    delta = mu_a - mu_b; nd2 = float(delta @ delta)
    u = delta / np.sqrt(nd2); mid = 0.5*(mu_a + mu_b)
    na = int(w * 20000); nb = 20000 - na
    A = Xa[rng.choice(len(Xa), na, replace=len(Xa) < na)]
    B = Xb[rng.choice(len(Xb), nb, replace=len(Xb) < nb)]
    m0 = np.concatenate([(A - mid) @ u, (B - mid) @ u]) * 2 / np.sqrt(nd2)
    return m0, nd2

def marginal_pdf(m0, tau):
    """p_tau(m) = histogram(m0) * N(0,tau), evaluated on GRID."""
    h, _ = np.histogram(m0, bins=np.append(GRID - DG/2, GRID[-1] + DG/2), density=True)
    k = np.exp(-GRID**2 / (2*tau)); k /= k.sum()
    p = np.convolve(h, k, mode='same')
    return np.maximum(p, 1e-300)

def stationary_points(p, merge=0.02):
    logp = np.log(p); d = np.gradient(logp, GRID)
    keep = p > 1e-4 * p.max()                       # ignore far tails
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    pts = []
    for i in idx:
        root = GRID[i] - d[i]*DG/(d[i+1]-d[i]) if d[i+1] != d[i] else GRID[i]
        pts.append((float(root), 'max' if d[i] > 0 else 'min'))
    # remove spurious max/min pairs created by numerical noise on flat regions
    changed = True
    while changed and len(pts) > 1:
        changed = False
        for a in range(len(pts)-1):
            if abs(pts[a][0] - pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]:
                del pts[a:a+2]; changed = True; break
    return pts

def bifurcation(m0, taus):
    return {tau: stationary_points(marginal_pdf(m0, tau)) for tau in taus}

def birth_jump(diag, taus):
    """Scan tau downward. At the first tau with >=2 maxima, measure the distance between the newly born
    max/min pair and the stable fixed point that existed at the previous tau.
    Pitchfork: the existing max itself splits -> distance ~ 0.  Fold: new pair appears elsewhere -> distance > 0."""
    ts = sorted(taus, reverse=True)
    prev_max = None
    for tau in ts:
        mx = sorted(p for p, k in diag[tau] if k == 'max')
        mn = [p for p, k in diag[tau] if k == 'min']
        if len(mx) >= 2 and prev_max is not None:
            c = mn[0] if mn else 0.5*(mx[0]+mx[-1])   # separatrix ~ where the new pair was born
            newmax = min(mx, key=lambda p: abs(p - c))   # the max nearest the separatrix = the new one
            return tau, abs(c - prev_max), mx, c, prev_max
        if len(mx) == 1:
            prev_max = mx[0]
    return None

## Run: digit pairs × weights × resolutions

In [ ]:
PAIRS = [(0, 1), (3, 8), (4, 9)]
WEIGHTS = [0.5, 0.8]
RES = [28, 14, 7]
TAUS = np.geomspace(0.05, 3.0, 60)
results = {}

print(f"{'pair':>6} {'w':>4} {'res':>4} {'d':>4} {'|Delta|^2':>10} {'tau_bif':>8} {'birth jump':>10} {'skew(m0)':>9}")
for pair in PAIRS:
    for res in RES:
        Xa = pool(X[Y == pair[0]], res); Xb = pool(X[Y == pair[1]], res)
        for w in WEIGHTS:
            m0, nd2 = order_param(Xa, Xb, w)
            diag = bifurcation(m0, TAUS)
            bj = birth_jump(diag, TAUS)
            skew = float(((m0 - m0.mean())**3).mean() / m0.std()**3)
            results[(pair, w, res)] = (m0, nd2, diag, bj)
            if bj:
                print(f"{str(pair):>6} {w:4.1f} {res:4d} {res*res:4d} {nd2:10.1f} {bj[0]:8.3f} {bj[1]:10.3f} {skew:9.3f}")
            else:
                print(f"{str(pair):>6} {w:4.1f} {res:4d} {res*res:4d} {nd2:10.1f}   never bimodal")

### Bifurcation diagrams at native resolution (28×28)

Blue = maxima (stable), red = minima (unstable). A pitchfork shows the two stable branches emerging from the red line at m≈0. A fold shows a stable branch and an unstable branch being born together at finite |m|.

In [ ]:
fig, axes = plt.subplots(len(WEIGHTS), len(PAIRS), figsize=(4.2*len(PAIRS), 3.6*len(WEIGHTS)), sharex=True)
for i, w in enumerate(WEIGHTS):
    for j, pair in enumerate(PAIRS):
        ax = axes[i, j]
        m0, nd2, diag, bj = results[(pair, w, 28)]
        for tau, pts in diag.items():
            for p, k in pts:
                ax.plot(tau, p, '.', ms=3, color='C0' if k == 'max' else 'C3')
        if bj: ax.axvline(bj[0], ls='--', color='k', lw=0.8)
        ax.set_xscale('log'); ax.set_title(f'digits {pair}, w={w}' + (f', birth |m|={bj[1]:.2f}' if bj else ''))
        ax.axhline(0, color='k', lw=0.4)
        if i == len(WEIGHTS)-1: ax.set_xlabel('tau = 4 sigma^2 / |Delta|^2')
        if j == 0: ax.set_ylabel('fixed point m*')
plt.tight_layout(); plt.show()

### Projected class distributions (why the pitchfork breaks)

If the two humps are not mirror images, the balanced case cannot be a symmetric pitchfork.

In [ ]:
fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.2*len(PAIRS), 3.2))
for ax, pair in zip(axes, PAIRS):
    m0, nd2, diag, bj = results[(pair, 0.5, 28)]
    ax.hist(m0, bins=120, density=True, alpha=0.7)
    ax.set_title(f'digits {pair}: projected m0 (balanced)'); ax.set_xlabel('m0')
plt.tight_layout(); plt.show()

### Resolution dependence of the diagram (pair (3,8), w=0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for res, col in zip(RES, ['C0', 'C1', 'C2']):
    m0, nd2, diag, bj = results[((3, 8), 0.5, res)]
    for tau, pts in diag.items():
        for p, k in pts:
            ax.plot(tau, p, '.', ms=3, color=col, alpha=1.0 if k == 'max' else 0.35)
    ax.plot([], [], 'o', color=col, label=f'res {res} (d={res*res}), |Delta|^2={nd2:.0f}')
ax.set_xscale('log'); ax.set_xlabel('tau'); ax.set_ylabel('m*'); ax.legend(); ax.set_title('(3,8) balanced: diagrams vs resolution')
plt.tight_layout(); plt.show()

## What to report back

1. The summary table: for **balanced** pairs, is `birth jump` clearly > 0 (fold) or ≈ 0 (pitchfork)? Reference values from the Step 1 control: 0 for the balanced mixture, ≈1.7 for w=0.8.
2. Whether the diagrams collapse across resolutions in tau (mean-field-like) or shift.
3. The projected histograms, and the skewness column: the hypothesis predicts that larger asymmetry between the two humps gives a larger birth jump.

If balanced real-data pairs already show folds, the paper's second claim is: **on real data the critical window is generically first-order-like, and the symmetric pitchfork is a measure-zero idealization.**